<img src="http://hilpisch.com/tpq_logo.png" width=350px align="right">

# Certificate Programs

**Mentoring Session**

## Stock Market Prediction

The Python Quants GmbH

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


In [ ]:
import numpy as np
import pandas as pd
from pylab import plt
plt.style.use('seaborn-v0_8')

## Getting Historical Data

In [ ]:
raw = pd.read_csv('http://hilpisch.com/aiif_eikon_eod_data.csv',
                  index_col=0, parse_dates=True).dropna()

In [ ]:
data = pd.DataFrame(raw['EUR='])

In [ ]:
data.columns = ['prices']

In [ ]:
data.info()

In [ ]:
data.plot(figsize=(10, 6));

In [ ]:
data['returns'] = np.log(data / data.shift(1))

In [ ]:
data.head()

In [ ]:
lags = 5

In [ ]:
cols = []

In [ ]:
for lag in range(1, lags+1):
    col = 'ret_%d' % lag
    data[col] = data['returns'].shift(lag)
    cols.append(col)

In [ ]:
data.head(10)

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.head()

## OLS Regression

In [ ]:
reg = np.linalg.lstsq(data[cols].values,
                      np.sign(data['returns'].values), rcond=-1)[0]

In [ ]:
reg

In [ ]:
pred = np.sign(np.dot(data[cols].values, reg))

In [ ]:
pred

In [ ]:
np.sign(data['returns'].values)

In [ ]:
data['ols_pred'] = pred

In [ ]:
c = np.sign(data['returns'] * data['ols_pred'])

In [ ]:
c.value_counts()

In [ ]:
c.value_counts()[1] / (c.value_counts().sum())

In [ ]:
data['ols_returns'] = data['returns'] * data['ols_pred']

In [ ]:
data[['returns', 'ols_returns']].cumsum().apply(np.exp).plot(figsize=(10, 6));

## Logistic Regression

In [ ]:
from sklearn import linear_model

In [ ]:
lm = linear_model.LogisticRegression(C=1000)

In [ ]:
lm.fit(data[cols], np.sign(data['returns']))

In [ ]:
data['log_pred'] = lm.predict(data[cols])

In [ ]:
data.head()

In [ ]:
data['log_returns'] = data['returns'] * data['log_pred']

In [ ]:
data[['returns', 'ols_returns', 'log_returns']].cumsum(
        ).apply(np.exp).plot(figsize=(10, 6));

## Deep Neural Network

In [ ]:
import os
import random
import tensorflow as tf

In [ ]:
def set_seeds(seed=100):
    os.environ['PYTHONHASHSEED'] = f'{seed}'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [ ]:
from keras.layers import Dense
from keras.models import Sequential
from keras.optimizers import Adam

In [ ]:
set_seeds()
model = Sequential()
model.add(Dense(64, activation='relu', input_dim=lags))
# model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy',
              optimizer=Adam(learning_rate=0.001),
              metrics=['accuracy'])

In [ ]:
mean = data[cols].mean()

In [ ]:
std = data[cols].std()

Sutton & Barto (2018): "Reinforcement Learning." (p. 226):

> It has long been known that ANN learning is easier if the network input is normalized, for example, by adjusting each input variable to have zero mean and unit variance.

See also https://www.jeremyjordan.me/batch-normalization/ for a nice discussion.

In [ ]:
data_ = data.copy()
data_[cols] = (data[cols] - mean) / std

In [ ]:
data_[cols].mean()

In [ ]:
data_[cols].std()

In [ ]:
model.fit(data_[cols], np.where(data['returns'] > 0, 1, 0),
          epochs=100, verbose=False)

In [ ]:
model.evaluate(data_[cols], np.where(data['returns'] > 0, 1, 0), steps=1)

In [ ]:
data['dnn_pred'] = model.predict_classes(data_[cols])
data['dnn_pred'] = np.where(data['dnn_pred'] > 0, 1, -1)

In [ ]:
data['dnn_returns'] = data['returns'] * data['dnn_pred']

In [ ]:
data[['returns', 'ols_returns',
      'log_returns', 'dnn_returns']].sum().apply(np.exp)

In [ ]:
data[['returns', 'ols_returns', 'log_returns', 'dnn_returns']].cumsum(
        ).apply(np.exp).plot(figsize=(10, 6));

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>